[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/whit3rabbit/qpclone/blob/main/Qwen3_TTS_Voice_Cloning_to_PIPER.ipynb)

# Qwen3-TTS Voice Cloning to Piper

Uses Qwen3-TTS to generate synthetic training audio from a short voice sample, then fine-tunes a Piper VITS model on that audio to produce a deployable offline TTS model.

## Before you start

### 1. Set up your Hugging Face token

A free Hugging Face API token is required to download the models.

1. Create a free account at [huggingface.co](https://huggingface.co/join)
2. Go to [huggingface.co/settings/tokens](https://huggingface.co/settings/tokens) and create a new token (the free `Read` access level is enough)
3. In Colab, click the **key icon** in the left sidebar (Secrets)
4. Click **"Add new secret"**
5. Set the name to `HF_TOKEN` and paste your token as the value
6. Toggle **"Notebook access"** on for this notebook

### 2. Prepare your setup

1. Have a **3+ second audio clip** of the voice you want to clone (clean speech, one speaker, no background noise)
2. **Change the project name** in the next cell
3. Run the cells in order -- **grant Google Drive permissions** when prompted (project files are stored there)
4. **Upload your audio file** when the upload cell runs

After the upload, click `Runtime > Run after` and walk away. Everything else is automated.

---

## Tips

- Record in a quiet room -- no echo, no fan noise, no music
- 3-5 seconds is the sweet spot for the voice sample
- Leave `REF_TEXT` blank and it will be auto-transcribed with Qwen3-ASR, or type your own for maximum accuracy
- 1,000 samples (default) is good for general use; 2,000+ for long-form narration

---

## Resuming After a Disconnect

All progress is saved to Drive. Re-run from the top and it picks up where it left off.

---

## Using Your Finished Model

```bash
pip install piper-tts
echo "Hello world" | piper -m your_project.onnx --output_file hello.wav
```

Piper runs on Linux, macOS, Windows, and Raspberry Pi.

---

## Troubleshooting

| Problem | Solution |
|---------|----------|
| **"CUDA out of memory"** | Switch to the 0.6B model in Advanced Settings |
| **Colab disconnected** | Just re-run -- resume is automatic |
| **Voice sounds robotic** | Check the auto-transcribed `REF_TEXT` is accurate, use a cleaner recording, or increase `NUM_SAMPLES` |
| **Training seems stuck** | Loss values decrease then level off. 1000 epochs takes a few hours. |
| **"401 Unauthorized" or model download fails** | Check that `HF_TOKEN` is set in Colab Secrets (key icon in sidebar) with notebook access enabled |

In [ ]:
# @title # 🎤 Voice Clone Settings
# @markdown ---
# @markdown ### Give your project a name and pick a language.
# @markdown The project folder will be created on Google Drive automatically.

PROJECT_NAME = "PROJECT_NAME"  # @param {type:"string"}
LANGUAGE = "English"  # @param ["English", "Chinese", "Japanese", "Korean", "German", "French", "Russian", "Portuguese", "Spanish", "Italian", "Auto"]
DRIVE_BASE_DIR = "/content/drive/MyDrive/Piper_Qwen_Projects"  # @param {type:"string"}

# @markdown ---
# @markdown ### Reference Audio
# @markdown Upload a **3–5 second** clean voice sample to clone.
# @markdown Optionally provide a transcript of the reference audio for better cloning quality.

REF_AUDIO_MODE = "upload_if_missing"  # @param ["upload_if_missing", "always_upload", "use_existing_only"]
REF_TEXT = ""  # @param {type:"string"}

# @markdown > 💡 `REF_TEXT`: transcript of your reference audio. Improves voice cloning quality.
# @markdown > Leave blank to auto-transcribe with Qwen3-ASR.

print(f"✅ Project: {PROJECT_NAME} ({LANGUAGE})")

In [ ]:
# @title 📦 Install Dependencies (run once)

!pip -q install -U qwen-tts datasets librosa soundfile torchaudio accelerate einops triton tiktoken tqdm
!pip -q install -U qwen-asr
!pip -q install -U onnxscript pathvalidate cloud-tpu-client
!sudo apt update && sudo apt install sox

# Check for CUDA
cuda_enabled = !nvidia-smi -L

# flash-attn is optional and can fail depending on Colab image; best-effort:
if "cuda" in cuda_enabled:
    if True:
        try:
            !pip -q install -U flash-attn --no-build-isolation
            print("flash-attn installed.")
        except Exception as e:
            print("flash-attn install failed; continuing without it.")

# Piper prerequisites
!sudo apt-get -qq update
!sudo apt-get -qq install espeak-ng

print("Dependency install complete.")

In [ ]:
# @title Imports + Helpers

# ---- Load HF token from Colab secrets ----
from google.colab import userdata
import os

# Prevent TensorFlow/JAX from pre-allocating GPU memory.
# These get imported as transitive deps of transformers/datasets and will
# silently hold ~1-2 GB of VRAM that Piper training needs later.
os.environ.setdefault('TF_FORCE_GPU_ALLOW_GROWTH', 'true')
os.environ.setdefault('XLA_PYTHON_CLIENT_PREALLOCATE', 'false')

try:
    hf_token = userdata.get('HF_TOKEN')
    os.environ['HF_TOKEN'] = hf_token
    os.environ['HUGGING_FACE_HUB_TOKEN'] = hf_token
    print("HF_TOKEN loaded from Colab secrets.")
except userdata.SecretNotFoundError:
    print("WARNING: HF_TOKEN not found in Colab secrets.")
    print("   Model downloads may fail. Click the key icon in the left sidebar")
    print("   to add a secret named HF_TOKEN with your Hugging Face API token.")
    print("   Get a free token at: https://huggingface.co/settings/tokens")
except Exception as e:
    print(f"WARNING: Could not load HF_TOKEN: {e}")

# ---- Imports ----
import re
import json
import time
import shutil
import hashlib
from typing import Optional, Dict, Any, List, Set

import torch
import librosa
import numpy as np
import soundfile as sf
from tqdm.auto import tqdm
from datasets import load_dataset

from google.colab import drive, files  # uncomment in Colab
from qwen_tts import Qwen3TTSModel  # uncomment in Colab

import transformers
transformers.logging.set_verbosity_error()

def slugify(name: str, max_len: int = 60) -> str:
    name = name.strip().lower()
    name = re.sub(r"[^a-z0-9._-]+", "_", name)
    name = re.sub(r"_+", "_", name).strip("_")
    return name[:max_len]


def ensure_dir(path: str) -> str:
    os.makedirs(path, exist_ok=True)
    return path


def read_json(path: str) -> Dict[str, Any]:
    if not os.path.exists(path):
        return {}
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def write_json(path: str, obj: Dict[str, Any]) -> None:
    tmp = path + ".tmp"
    with open(tmp, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, ensure_ascii=False)
    os.replace(tmp, path)


def sha256_file(path: str) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()


def pick_torch_dtype(dtype_str: str):
    return {
        "bfloat16": torch.bfloat16,
        "float16": torch.float16,
        "float32": torch.float32,
    }.get(dtype_str, torch.bfloat16)


def load_existing_metadata(metadata_path: str) -> tuple[Dict[str, str], int]:
    """
    Load existing metadata.csv and return:
      - dict mapping basename -> text (for skip-existing checks)
      - highest numeric ID found (for continuing sequential numbering)
    """
    entries: Dict[str, str] = {}
    max_numeric_id = 0
    if not os.path.exists(metadata_path):
        return entries, max_numeric_id
    with open(metadata_path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            parts = line.split("|", 1)
            if len(parts) == 2:
                basename, text = parts[0].strip(), parts[1].strip()
                entries[basename] = text
                # Track highest numeric ID
                try:
                    num = int(basename)
                    max_numeric_id = max(max_numeric_id, num)
                except ValueError:
                    pass
    return entries, max_numeric_id


def validate_wav(wav_path: str, min_duration_s: float = 0.3) -> bool:
    """Quick validation that a wav file is readable and has some duration."""
    try:
        info = sf.info(wav_path)
        return info.duration >= min_duration_s
    except Exception:
        return False


print("✅ Imports + helpers ready.")

In [ ]:
# ============================================================
# @title 💾 Mount Drive + Project Paths
# ============================================================

drive.mount("/content/drive")

project_slug = slugify(PROJECT_NAME)
PROJECT_DIR = ensure_dir(os.path.join(DRIVE_BASE_DIR, project_slug))

REF_DIR = ensure_dir(os.path.join(PROJECT_DIR, "reference"))
DATASET_DIR = ensure_dir(os.path.join(PROJECT_DIR, "dataset"))
WAVS_DIR = ensure_dir(os.path.join(DATASET_DIR, "wavs"))
PIPER_DIR = ensure_dir(os.path.join(PROJECT_DIR, "piper"))
OUTPUTS_DIR = ensure_dir(os.path.join(PROJECT_DIR, "outputs"))

PROJECT_JSON_PATH = os.path.join(PROJECT_DIR, "project.json")
METADATA_PATH = os.path.join(DATASET_DIR, "metadata.csv")

print("PROJECT_DIR:", PROJECT_DIR)

In [ ]:
# ============================================================
# @title Init Project State
# ============================================================

state = read_json(PROJECT_JSON_PATH)

state.setdefault("project_name", PROJECT_NAME)
state.setdefault("project_slug", project_slug)
state.setdefault("created_at_unix", int(time.time()))
state.setdefault("language", LANGUAGE)
state.setdefault("ref_audio", {})
state.setdefault("dataset", {})
state.setdefault("piper", {})
state.setdefault("history", [])

write_json(PROJECT_JSON_PATH, state)
print("Project state initialized.")


In [ ]:
# ============================================================
# @title Reference Audio (upload/reuse)
# ============================================================

def find_existing_ref_audio() -> Optional[str]:
    ref_path = state.get("ref_audio", {}).get("path")
    if ref_path and os.path.exists(ref_path):
        return ref_path
    if os.path.exists(REF_DIR):
        for fn in sorted(os.listdir(REF_DIR)):
            if fn.lower().endswith((".wav", ".mp3", ".flac", ".m4a", ".ogg")):
                return os.path.join(REF_DIR, fn)
    return None


existing_ref = find_existing_ref_audio()

if REF_AUDIO_MODE == "use_existing_only":
    if not existing_ref:
        raise FileNotFoundError(
            "No existing reference audio found, and REF_AUDIO_MODE=use_existing_only."
        )
    ref_audio_path = existing_ref
    print("Using existing reference audio:", ref_audio_path)

elif REF_AUDIO_MODE == "upload_if_missing":
    if existing_ref:
        ref_audio_path = existing_ref
        print("Using existing reference audio:", ref_audio_path)
    else:
        print("Upload a 3-5 second clean sample (.wav/.mp3/etc).")
        up = files.upload()
        if not up:
            raise RuntimeError("Upload canceled/no file provided.")
        src_name = next(iter(up.keys()))
        dst_name = re.sub(r"[^A-Za-z0-9._-]+", "_", os.path.basename(src_name).replace(" ", "_")).strip("_")
        if not dst_name:
            dst_name = f"audio_{int(time.time())}.wav"
        ref_audio_path = os.path.join(REF_DIR, dst_name)
        shutil.move(src_name, ref_audio_path)
        print("Saved:", ref_audio_path)

elif REF_AUDIO_MODE == "always_upload":
    print("Upload a 3-5 second clean sample (.wav/.mp3/etc).")
    up = files.upload()
    if not up:
        raise RuntimeError("Upload canceled/no file provided.")
    src_name = next(iter(up.keys()))
    dst_name = re.sub(r"[^A-Za-z0-9._-]+", "_", os.path.basename(src_name).replace(" ", "_")).strip("_")
    if not dst_name:
        dst_name = f"audio_{int(time.time())}.wav"
    ref_audio_path = os.path.join(REF_DIR, dst_name)
    shutil.move(src_name, ref_audio_path)
    print("Saved:", ref_audio_path)

else:
    raise ValueError(f"Invalid REF_AUDIO_MODE: {REF_AUDIO_MODE}")

# --- Auto-trim long reference audio ---
# Qwen3-TTS can hallucinate or loop endlessly with long reference prompts,
# causing dataset generation to hang. Trim to a safe maximum duration.
MAX_REF_DURATION = 10.0
try:
    audio_data, sr = librosa.load(ref_audio_path, sr=None)
    duration = librosa.get_duration(y=audio_data, sr=sr)

    if duration > MAX_REF_DURATION:
        print(f"\nReference audio is too long ({duration:.1f}s). Auto-trimming to first {MAX_REF_DURATION:.0f}s to prevent generation errors...")
        audio_data = audio_data[:int(MAX_REF_DURATION * sr)]

        # Save the trimmed audio, enforcing standard 16-bit WAV
        sf.write(ref_audio_path, audio_data, sr, subtype='PCM_16')

        # If user typed a manual text, but we cut the audio, the text no longer matches.
        # Clear it so the ASR auto-transcriber runs on the exact trimmed audio slice.
        if REF_TEXT.strip():
            print("Audio was trimmed -- clearing manual REF_TEXT so the truncated audio can be auto-transcribed accurately.")
            REF_TEXT = ""
except Exception as e:
    print(f"\nWarning: Could not check/auto-trim reference audio: {e}")
# --- End auto-trim ---

ref_hash = sha256_file(ref_audio_path)
state["ref_audio"] = {
    "path": ref_audio_path,
    "sha256": ref_hash,
    "ref_text": REF_TEXT,
    "ref_text_source": "manual" if REF_TEXT.strip() else "pending",
    "recorded_at_unix": int(time.time()),
}
state["history"].append({
    "t": int(time.time()),
    "event": "ref_audio_set",
    "path": ref_audio_path,
    "sha256": ref_hash,
})
write_json(PROJECT_JSON_PATH, state)
print("Reference audio ready:", ref_audio_path)
if REF_TEXT:
    print(f"   Reference transcript: \"{REF_TEXT[:80]}{'...' if len(REF_TEXT) > 80 else ''}\"")
else:
    print("   REF_TEXT is blank -- will auto-transcribe with Qwen3-ASR.")

---

**You can walk away now.** Run all remaining cells (`Runtime > Run after`) and come back when training is done. Everything below uses sensible defaults.

---

In [ ]:
# @title # 📝 Dataset & Generation
# @markdown ---
# @markdown ### How many voice samples to generate?
# @markdown More samples = better quality voice. **500–2000** recommended for long speeches.

NUM_SAMPLES = 1000  # @param {type:"slider", min:100, max:3000, step:100}
SKIP_EXISTING_WAVS = True  # @param {type:"boolean"}

# @markdown ---
# @markdown ### Text source
# @markdown Where to pull transcripts from. Default uses LJ Speech (13k English sentences).

TEXT_DATASET_ID = "MikhailT/lj-speech"  # @param {type:"string"}
TEXT_DATASET_SPLIT = "full"             # @param {type:"string"}
TEXT_FIELD = "normalized_text"          # @param {type:"string"}

# @markdown ---
# @markdown ### Text filtering

MAX_TEXT_CHARS = 150  # @param {type:"slider", min:50, max:500, step:10}
MIN_TEXT_CHARS = 10   # @param {type:"slider", min:1, max:50, step:1}
START_INDEX = 0       # @param {type:"integer"}

# @markdown ---
# @markdown ### Audio duration cap
# @markdown Clips longer than this are discarded after generation to prevent OOM during training.

MAX_DURATION_S = 10.0  # @param {type:"number"}

print(f"✅ Will generate {NUM_SAMPLES} samples from '{TEXT_DATASET_ID}'")

In [ ]:
# @title # Piper Training Settings
# @markdown ---
# @markdown ### Training duration & checkpoints
# @markdown More epochs = better quality but longer training time.

PIPER_MAX_EPOCHS = 4000  # @param {type:"slider", min:100, max:10000, step:100}
PIPER_BATCH_SIZE = 40    # @param {type:"slider", min:4, max:64, step:4}
PIPER_CHECKPOINT_EVERY = 10  # @param {type:"slider", min:5, max:100, step:5}
PIPER_RESUME = True      # @param {type:"boolean"}

# @markdown ---
# @markdown ### Validation
PIPER_VALIDATION_SPLIT = 0.05  # @param {type:"slider", min:0.01, max:0.2, step:0.01}

# @markdown ---
# @markdown ### Pretrained checkpoint (fine-tuning)
# @markdown Path within the `rhasspy/piper-checkpoints` HuggingFace repo.
# @markdown Format: `lang/region/speaker/quality` (e.g. `en/en_US/lessac/medium`).
# @markdown Set to `none` to train from scratch (not recommended with < 5000 samples).
# @markdown
# @markdown Browse available checkpoints:
# @markdown [huggingface.co/rhasspy/piper-checkpoints](https://huggingface.co/rhasspy/piper-checkpoints)
PIPER_PRETRAINED_CHECKPOINT = "en/en_US/lessac/medium"  # @param {type:"string"}

print(f"Piper: {PIPER_MAX_EPOCHS} epochs, batch size {PIPER_BATCH_SIZE}")
if PIPER_PRETRAINED_CHECKPOINT.lower() != "none":
    print(f"   Fine-tuning from: {PIPER_PRETRAINED_CHECKPOINT}")
else:
    print(f"   Training from scratch (no pretrained checkpoint)")

In [ ]:
# @title Advanced Settings (expand if needed) { display-mode: "form" }
# @markdown ---
# @markdown ### Model & Hardware
# @markdown *You probably don't need to change these.*
# @markdown
# @markdown **1.7B-Base**: Higher quality, more VRAM (~8GB). **0.6B-Base**: Faster, less VRAM (~4GB).

QWEN_MODEL_ID = "Qwen/Qwen3-TTS-12Hz-1.7B-Base"  # @param ["Qwen/Qwen3-TTS-12Hz-1.7B-Base", "Qwen/Qwen3-TTS-12Hz-0.6B-Base"]
DEVICE = "cuda"        # @param ["cuda", "cpu"]
DTYPE = "bfloat16"     # @param ["bfloat16", "float16", "float32"]
USE_FLASH_ATTN = False  # @param {type:"boolean"}
PIPER_PRECISION = 16    # @param [16, 32]

# @markdown ---
# @markdown ### Audio
# @markdown Qwen3-TTS outputs at 24kHz. Piper typically expects 22050Hz.
# @markdown Audio is automatically resampled to the output sample rate.
OUTPUT_SAMPLE_RATE = 22050    # @param [16000, 22050, 44100]

# @markdown ---
# @markdown ### Dataset field mapping
# @markdown *Only change if using a custom text dataset.*
ID_FIELD = "id"  # @param {type:"string"}

print("Advanced settings loaded.")
print(f"   Model: {QWEN_MODEL_ID}")
print(f"   Device: {DEVICE} | Dtype: {DTYPE} | Precision: {PIPER_PRECISION}")
print(f"   Output SR: {OUTPUT_SAMPLE_RATE} Hz")

In [ ]:
# ============================================================
# @title Save Config to Project State
# ============================================================

state = read_json(PROJECT_JSON_PATH)

state["qwen_model_id"] = QWEN_MODEL_ID
state["last_config"] = {
    "LANGUAGE": LANGUAGE,
    "QWEN_MODEL_ID": QWEN_MODEL_ID,
    "NUM_SAMPLES": NUM_SAMPLES,
    "START_INDEX": START_INDEX,
    "OUTPUT_SAMPLE_RATE": OUTPUT_SAMPLE_RATE,
    "DEVICE": DEVICE,
    "DTYPE": DTYPE,
    "PIPER_PRETRAINED_CHECKPOINT": PIPER_PRETRAINED_CHECKPOINT,
}

write_json(PROJECT_JSON_PATH, state)
print("Config saved to project state.")


In [ ]:
# ============================================================
# @title 🎤 Auto-transcribe Reference Audio (Qwen3-ASR)
# ============================================================
# If REF_TEXT is blank, use Qwen3-ASR to transcribe the reference audio.
# Manual REF_TEXT is always used as-is. On resume, reuses saved transcript.

import gc

state = read_json(PROJECT_JSON_PATH)
ref_text_source = state.get("ref_audio", {}).get("ref_text_source", "pending")

if REF_TEXT.strip():
    # Branch 1: User provided REF_TEXT manually -- skip ASR entirely
    print("REF_TEXT provided manually -- skipping ASR.")
    print(f"  Transcript: \"{REF_TEXT[:80]}{'...' if len(REF_TEXT) > 80 else ''}\"")

elif ref_text_source == "asr":
    # Branch 2: Resume path -- ASR already ran in a previous session
    saved_text = state.get("ref_audio", {}).get("ref_text", "")
    if saved_text.strip():
        REF_TEXT = saved_text
        print("Reusing ASR transcript from previous session:")
        print(f"  Transcript: \"{REF_TEXT[:80]}{'...' if len(REF_TEXT) > 80 else ''}\"")
    else:
        print("Previous ASR produced empty transcript -- using x-vector-only mode.")

else:
    # Branch 3: Run ASR to transcribe reference audio
    print("Auto-transcribing reference audio with Qwen3-ASR-0.6B...")

    from qwen_asr import Qwen3ASRModel

    asr_model_id = "Qwen/Qwen3-ASR-0.6B"
    asr_model = Qwen3ASRModel.from_pretrained(
        asr_model_id,
        dtype=torch.bfloat16,
        device_map="cuda:0",
        max_new_tokens=256,
    )
    print(f"  ASR model loaded: {asr_model_id}")

    # Map notebook LANGUAGE to ASR language hint (None = auto-detect)
    _lang_map = {
        "English": "English", "Chinese": "Chinese", "Japanese": "Japanese",
        "Korean": "Korean", "German": "German", "French": "French",
        "Russian": "Russian", "Portuguese": "Portuguese", "Spanish": "Spanish",
        "Italian": "Italian", "Auto": None,
    }
    asr_language = _lang_map.get(LANGUAGE)

    results = asr_model.transcribe(
        audio=ref_audio_path,
        language=asr_language,
    )
    transcript = results[0].text.strip() if results and results[0].text else ""

    # Unload ASR model completely before TTS loads
    del asr_model
    del results
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.synchronize()
    print("  ASR model unloaded, GPU memory freed.")

    if transcript:
        REF_TEXT = transcript
        state["ref_audio"]["ref_text"] = REF_TEXT
        state["ref_audio"]["ref_text_source"] = "asr"
        write_json(PROJECT_JSON_PATH, state)
        print(f"  Transcript: \"{REF_TEXT[:80]}{'...' if len(REF_TEXT) > 80 else ''}\"")
        print("  Saved to project state (ref_text_source: asr).")
    else:
        state["ref_audio"]["ref_text_source"] = "none"
        write_json(PROJECT_JSON_PATH, state)
        print("  ASR returned empty transcript -- falling back to x-vector-only mode.")

In [ ]:
# ============================================================
# @title 🤖 Load Qwen3-TTS
# ============================================================

torch_dtype = pick_torch_dtype(DTYPE)
use_cuda = (DEVICE == "cuda") and torch.cuda.is_available()
device_map = "cuda:0" if use_cuda else "cpu"

# Attention implementation
if USE_FLASH_ATTN and DTYPE in ("bfloat16", "float16"):
    attn_impl = "flash_attention_2"
else:
    attn_impl = "sdpa"  # scaled dot-product attention (PyTorch native)

print("Loading:", QWEN_MODEL_ID)
print("  device_map:", device_map)
print("  dtype:", torch_dtype)
print("  attn_implementation:", attn_impl)

model = Qwen3TTSModel.from_pretrained(
    QWEN_MODEL_ID,
    device_map=device_map,
    dtype=torch_dtype,
    attn_implementation=attn_impl,
)

print("✅ Qwen3-TTS model loaded.")

In [ ]:
# ============================================================
# @title 🔗 Pre-compute Voice Clone Prompt
# ============================================================
# Pre-computing the prompt extracts speaker features from the reference
# audio ONCE, then reuses them for every generation call.
# This is much faster than passing ref_audio to every call.

print("Pre-computing voice clone prompt from reference audio...")

use_xvector_only = not bool(REF_TEXT.strip())

voice_clone_prompt = model.create_voice_clone_prompt(
    ref_audio=ref_audio_path,
    ref_text=REF_TEXT if REF_TEXT.strip() else None,
    x_vector_only_mode=use_xvector_only,
)

print("✅ Voice clone prompt ready.")
if use_xvector_only:
    print("   Mode: x-vector only (no ref_text provided)")
else:
    print("   Mode: full prompt (ref_text + audio features)")

In [ ]:
# ============================================================
# @title 📝 Load Text Dataset
# ============================================================

dataset_id = TEXT_DATASET_ID
print(f"Loading dataset: {dataset_id} [{TEXT_DATASET_SPLIT}]")

ds = load_dataset(dataset_id, split=TEXT_DATASET_SPLIT)

# ---- Field validation ----
available_fields = ds.column_names

if TEXT_FIELD not in available_fields:
    raise ValueError(
        f"TEXT_FIELD '{TEXT_FIELD}' not found. Available: {available_fields}"
    )

use_index_as_id = ID_FIELD not in available_fields
if use_index_as_id:
    print(f"⚠️ ID_FIELD '{ID_FIELD}' not found. Using row index instead.")

# ---- Slice subset ----
total_len = len(ds)
if START_INDEX >= total_len:
    raise ValueError(f"START_INDEX {START_INDEX} >= dataset length {total_len}")

end = min(total_len, START_INDEX + NUM_SAMPLES)
subset = ds.select(range(START_INDEX, end))

texts: List[str] = []
for idx, row in enumerate(subset):
    raw_text = str(row[TEXT_FIELD]).strip().replace("\n", " ")
    text = raw_text[:MAX_TEXT_CHARS]

    if len(text) < MIN_TEXT_CHARS:
        continue

    texts.append(text)

print(f"✅ Loaded {len(texts)} usable texts (from index {START_INDEX}, "
      f"skipped {(end - START_INDEX) - len(texts)} too-short entries).")

if len(texts) < 50:
    print("⚠️ WARNING: Fewer than 50 samples. Piper training may underfit.")

state["dataset"].update({
    "text_dataset_id": dataset_id,
    "text_split": TEXT_DATASET_SPLIT,
    "text_field": TEXT_FIELD,
    "dataset_total_len": total_len,
    "requested_num_samples": NUM_SAMPLES,
    "effective_num_samples": len(texts),
    "output_sample_rate": OUTPUT_SAMPLE_RATE,
})
write_json(PROJECT_JSON_PATH, state)

## Piper

In [ ]:
# ============================================================
# @title Generate Piper Training Samples (wavs + metadata.csv)
# ============================================================
#
# Piper expects:
#   dataset/
#     metadata.csv   -- lines of: <wav_basename_no_ext>|<transcript>
#     wavs/
#       1.wav
#       2.wav
#       ...
#
# Qwen3-TTS outputs at 24kHz. We resample to OUTPUT_SAMPLE_RATE (default 22050).
# Audio is peak-normalized to 0.95 and written as 16-bit PCM, which is what
# Piper's VITS mel processing expects. Without this, float32 WAVs with values
# outside [-1, 1] cause "audio amplitude out of range" warnings during training.
# ============================================================

# Load existing metadata to support resumption
existing_meta, max_existing_id = load_existing_metadata(METADATA_PATH)
print(f"Existing metadata entries: {len(existing_meta)}, "
      f"highest ID: {max_existing_id}")

# Build reverse lookup: text -> basename, to skip already-generated texts
existing_texts: Set[str] = set(existing_meta.values())

# Next sequential ID
next_id = max_existing_id + 1

generated = 0
skipped = 0
errors = 0

# Open metadata for appending
with open(METADATA_PATH, "a", encoding="utf-8") as meta_f:
    for text in tqdm(texts, desc="Generating TTS samples"):
        # --- Skip if this exact text already has a wav ---
        if SKIP_EXISTING_WAVS and text in existing_texts:
            skipped += 1
            continue

        # Sequential naming: 1.wav, 2.wav, ...
        wav_basename = str(next_id)
        wav_path = os.path.join(WAVS_DIR, f"{wav_basename}.wav")

        try:
            # ============================================================
            # Qwen3-TTS: generate_voice_clone()
            # Returns: (wavs: list[np.ndarray], sr: int)
            # Output sr is always 24000 Hz
            # ============================================================
            wavs, sr = model.generate_voice_clone(
                text=text,
                language=LANGUAGE,
                voice_clone_prompt=voice_clone_prompt,
            )

            audio = wavs[0]  # numpy array, 1D, float

            # Ensure 1D float32
            audio = np.asarray(audio, dtype=np.float32).squeeze()
            if audio.ndim > 1:
                audio = audio[0]

            # Resample from Qwen output (24kHz) to Piper target
            if sr != OUTPUT_SAMPLE_RATE:
                audio = librosa.resample(
                    audio, orig_sr=sr, target_sr=OUTPUT_SAMPLE_RATE
                )

            # Validate: audio should have reasonable length
            duration_s = len(audio) / OUTPUT_SAMPLE_RATE
            if duration_s < 0.2:
                print(f"Warning: Skipping ID {wav_basename}: audio too short "
                      f"({duration_s:.2f}s)")
                errors += 1
                continue

            if duration_s > MAX_DURATION_S:
                print(f"Warning: Skipping ID {wav_basename}: audio too long "
                      f"({duration_s:.2f}s, max {MAX_DURATION_S}s)")
                errors += 1
                continue

            # --- Audio normalization for Piper compatibility ---
            # Qwen output can exceed [-1.0, 1.0]. Piper's VITS mel processing
            # expects normalized 16-bit PCM. We peak-normalize to 0.95 to avoid
            # clipping distortion during training.
            peak = np.max(np.abs(audio))
            if peak < 1e-6:
                # Near-silent output -- skip this sample
                print(f"Warning: Skipping ID {wav_basename}: near-silent audio "
                      f"(peak={peak:.2e})")
                errors += 1
                continue
            audio = audio * (0.95 / peak)
            audio = np.clip(audio, -1.0, 1.0)

            # Write wav as 16-bit PCM (required by Piper's mel spectrogram processing)
            sf.write(wav_path, audio, OUTPUT_SAMPLE_RATE, subtype='PCM_16')

            # Validate the written file
            if not validate_wav(wav_path):
                print(f"Warning: Written wav failed validation: {wav_path}")
                os.remove(wav_path)
                errors += 1
                continue

            # Write metadata line and flush immediately (crash safety)
            meta_f.write(f"{wav_basename}|{text}\n")
            meta_f.flush()

            # Track state
            existing_texts.add(text)
            existing_meta[wav_basename] = text
            generated += 1
            next_id += 1

        except Exception as e:
            print(f"Warning: Error generating ID {wav_basename}: {e}")
            errors += 1
            # Don't increment next_id on error -- reuse this slot
            continue

# Update project state
state["history"].append({
    "t": int(time.time()),
    "event": "dataset_generate",
    "generated": generated,
    "skipped": skipped,
    "errors": errors,
})
state["dataset"]["total_generated"] = len(existing_meta)
write_json(PROJECT_JSON_PATH, state)

print(f"\nGeneration complete.")
print(f"   Generated: {generated}")
print(f"   Skipped (already existed): {skipped}")
print(f"   Errors: {errors}")
print(f"   Total in dataset: {len(existing_meta)}")
print(f"   Dataset dir: {DATASET_DIR}")
print(f"   Metadata: {METADATA_PATH}")

# ---- Verify final dataset integrity ----
final_meta, _ = load_existing_metadata(METADATA_PATH)
wav_files = {f.replace(".wav", "") for f in os.listdir(WAVS_DIR) if f.endswith(".wav")}
meta_keys = set(final_meta.keys())

orphan_wavs = wav_files - meta_keys
orphan_meta = meta_keys - wav_files

if orphan_wavs:
    print(f"Warning: {len(orphan_wavs)} orphan wav files (no metadata entry): "
          f"{list(orphan_wavs)[:5]}...")
if orphan_meta:
    print(f"Warning: {len(orphan_meta)} orphan metadata entries (no wav file): "
          f"{list(orphan_meta)[:5]}...")

matched = wav_files & meta_keys
print(f"Verified: {len(matched)} matched wav+metadata pairs.")

RECOMMENDED_MIN = 200
if len(matched) < RECOMMENDED_MIN:
    print(f"\nWARNING: {len(matched)} samples is below the recommended "
          f"minimum of {RECOMMENDED_MIN} for Piper fine-tuning. "
          f"For long speeches, aim for 500-2000+ samples with varied text.")
else:
    total_duration_est = len(matched) * 4.0
    print(f"Estimated total audio: ~{total_duration_est/60:.0f} minutes "
          f"({total_duration_est/3600:.1f} hours)")

In [ ]:
# ============================================================
# @title 🧪 Setup Piper (clone repo + fix dependencies)
# ============================================================
# @markdown The official Piper repo was archived Oct 2025 and its
# @markdown `requirements.txt` is broken on modern Colab (Python 3.12).
# @markdown This cell pins known-working versions of all dependencies.

import os
import subprocess
import sys

PIPER_REPO_DIR = os.path.join(PIPER_DIR, "piper")

# ---- Clone if needed ----
if not os.path.exists(PIPER_REPO_DIR):
    !git clone -q https://github.com/rhasspy/piper.git "{PIPER_REPO_DIR}"
    print("✅ Piper repo cloned.")
else:
    print("Piper repo already exists:", PIPER_REPO_DIR)

# ---- Check Python version ----
py_version = sys.version_info
print(f"Python version: {py_version.major}.{py_version.minor}.{py_version.micro}")

# ---- Install piper-phonemize ----
# The official piper-phonemize on PyPI only has wheels for Python 3.9-3.11.
# For Python 3.12+, we use piper-phonemize-cross which has broader wheel support.
# For 3.10-3.11, the official package works fine.

print("\n📦 Installing piper-phonemize...")
if py_version.minor >= 12:
    # piper-phonemize-cross is a community rebuild with wheels for 3.12+
    !pip -q install piper-phonemize-cross
    # Create a compatibility shim so imports still work
    try:
        import piper_phonemize
        print("   piper_phonemize already importable.")
    except ImportError:
        print("   Setting up piper_phonemize compatibility shim...")
        import piper_phonemize_cross
        # Some code imports piper_phonemize directly
        sys.modules['piper_phonemize'] = piper_phonemize_cross
else:
    !pip -q install piper-phonemize==1.1.0

# ---- Install pytorch-lightning (pinned to 1.9.x for Piper compat) ----
# Piper's code uses `from pytorch_lightning import Trainer` (old package name)
# pytorch-lightning 1.9.x is the last version that uses this import path
# and is compatible with modern PyTorch 2.x
print("\n📦 Installing pytorch-lightning...")
!pip -q install "pytorch-lightning>=1.9.0,<2.0.0" torchmetrics

# ---- Install remaining piper training deps ----
# Do NOT use requirements.txt — it has broken pins
print("\n📦 Installing other training dependencies...")
!pip -q install cython onnxruntime onnxsim

# ---- Source directory for piper_train ----
train_src = os.path.join(PIPER_REPO_DIR, "src", "python")

# ---- PyTorch 2.6+ compatibility: checkpoint loading ----
# PyTorch 2.6 defaults torch.load to weights_only=True, rejecting
# pathlib.PosixPath objects found in Piper pretrained checkpoints.
# Patch piper_train/__init__.py so every submodule gets the fix.
print("\n🔧 Patching for PyTorch 2.6+ checkpoint compatibility...")
_init_py = os.path.join(train_src, "piper_train", "__init__.py")
_PATCH_MARKER = "# PATCH: PyTorch 2.6+ weights_only"
_PATCH_CODE = (
    "# PATCH: PyTorch 2.6+ weights_only\n"
    "import pathlib as _pathlib\n"
    "try:\n"
    "    import torch.serialization\n"
    "    torch.serialization.add_safe_globals([_pathlib.PosixPath, _pathlib.WindowsPath])\n"
    "except (ImportError, AttributeError):\n"
    "    pass  # older PyTorch, not needed\n"
)
_existing = ""
if os.path.exists(_init_py):
    with open(_init_py, "r") as f:
        _existing = f.read()
if _PATCH_MARKER not in _existing:
    with open(_init_py, "w") as f:
        f.write(_PATCH_CODE + "\n" + _existing)
    print("   Applied to piper_train/__init__.py")
else:
    print("   Already patched")

# ---- Build monotonic_align Cython extension ----
# This is inside piper_train/vits/monotonic_align/, NOT src/python/monotonic_align/
print("\n🔨 Building monotonic_align...")

# Method 1: Use the build script if it exists
build_script = os.path.join(train_src, "build_monotonic_align.sh")
if os.path.exists(build_script):
    result = subprocess.run(
        ["bash", build_script],
        cwd=train_src,
        capture_output=True, text=True
    )
    if result.returncode == 0:
        print("   ✅ monotonic_align built via build_monotonic_align.sh")
    else:
        print(f"   ⚠️ build script failed: {result.stderr[:200]}")
        print("   Trying manual build...")

# Method 2: Build directly from the Cython source
mon_align_dir = os.path.join(
    train_src, "piper_train", "vits", "monotonic_align"
)
if not os.path.exists(mon_align_dir):
    # Try alternate location
    mon_align_dir = os.path.join(train_src, "monotonic_align")

if os.path.exists(mon_align_dir):
    setup_py = os.path.join(mon_align_dir, "setup.py")
    if os.path.exists(setup_py):
        result = subprocess.run(
            [sys.executable, "setup.py", "build_ext", "--inplace"],
            cwd=mon_align_dir,
            capture_output=True, text=True
        )
        if result.returncode == 0:
            print(f"   ✅ monotonic_align built from {mon_align_dir}")
        else:
            print(f"   ⚠️ Build failed: {result.stderr[:300]}")
    else:
        print(f"   ⚠️ No setup.py found in {mon_align_dir}")
else:
    print(f"   ⚠️ monotonic_align dir not found. Checked:")
    print(f"      {os.path.join(train_src, 'piper_train', 'vits', 'monotonic_align')}")
    print(f"      {os.path.join(train_src, 'monotonic_align')}")
    print(f"   Training may fall back to a slower Python implementation.")

# ---- Verify critical imports ----
print("\n🔍 Verifying imports...")
checks = {
    "pytorch_lightning": "import pytorch_lightning; print(f'  pytorch_lightning {pytorch_lightning.__version__}')",
    "piper_phonemize": "import piper_phonemize; print('  piper_phonemize OK')",
    "onnxruntime": "import onnxruntime; print(f'  onnxruntime {onnxruntime.__version__}')",
}
all_ok = True
for name, cmd in checks.items():
    try:
        exec(cmd)
    except ImportError as e:
        print(f"  ❌ {name}: {e}")
        all_ok = False

if all_ok:
    print("\n✅ Piper setup complete — all dependencies verified.")
else:
    print("\n⚠️ Some imports failed. Training may not work. Check errors above.")


In [ ]:
# ============================================================
# @title Download Pretrained Piper Checkpoint (for fine-tuning)
# ============================================================
# @markdown Downloads a pretrained Piper VITS checkpoint from HuggingFace.
# @markdown Fine-tuning from a pretrained model produces much better results
# @markdown with small datasets (< 5000 samples) compared to training from scratch.

import os
from huggingface_hub import list_repo_tree, hf_hub_download

PRETRAINED_DIR = ensure_dir(os.path.join(PIPER_DIR, "pretrained"))

# Language-to-checkpoint mapping for auto-matching
LANG_TO_CHECKPOINT = {
    "English": "en/en_US/lessac/medium",
    "Chinese": "zh/zh_CN/huayan/medium",
    "German":  "de/de_DE/thorsten/medium",
    "French":  "fr/fr_FR/siwis/medium",
    "Russian": "ru/ru_RU/irina/medium",
    "Portuguese": "pt/pt_BR/faber/medium",
    "Spanish": "es/es_ES/davefx/medium",
    "Italian": "it/it_IT/riccardo/medium",
}
# Languages without Piper checkpoints -- fall back to English
LANGS_WITHOUT_CHECKPOINT = {"Japanese", "Korean"}

pretrained_ckpt_path = None

if PIPER_PRETRAINED_CHECKPOINT.lower() == "none":
    print("Pretrained checkpoint disabled -- will train from scratch.")
    print("NOTE: Training from scratch needs 5000+ samples for decent results.")
else:
    ckpt_subpath = PIPER_PRETRAINED_CHECKPOINT

    # Auto-match language if user left the default English checkpoint
    # but selected a different language
    default_en = "en/en_US/lessac/medium"
    if ckpt_subpath == default_en and LANGUAGE != "English":
        if LANGUAGE in LANGS_WITHOUT_CHECKPOINT:
            print(f"WARNING: No Piper checkpoint available for {LANGUAGE}.")
            print(f"   Using English checkpoint ({default_en}) for cross-language fine-tuning.")
            print(f"   Results may be degraded. Consider providing more training samples.")
        elif LANGUAGE in LANG_TO_CHECKPOINT:
            ckpt_subpath = LANG_TO_CHECKPOINT[LANGUAGE]
            print(f"Auto-switched checkpoint to match language '{LANGUAGE}': {ckpt_subpath}")
        else:
            print(f"WARNING: Unknown language '{LANGUAGE}', no checkpoint mapping found.")
            print(f"   Keeping default English checkpoint: {default_en}")

    repo_id = "rhasspy/piper-checkpoints"
    ckpt_prefix = ckpt_subpath + "/"

    print(f"\nLooking for checkpoint in: {repo_id} / {ckpt_subpath}")

    # Find the .ckpt file dynamically (filenames include epoch/step numbers
    # that change between releases, so we can't hardcode them)
    try:
        ckpt_filename = None
        for entry in list_repo_tree(repo_id, path_in_repo=ckpt_subpath, repo_type="dataset"):
            name = entry.rfilename if hasattr(entry, 'rfilename') else str(entry)
            if name.endswith(".ckpt"):
                ckpt_filename = name
                break

        if ckpt_filename is None:
            print(f"ERROR: No .ckpt file found in {repo_id}/{ckpt_subpath}")
            print("   Check that the checkpoint path is correct.")
            print(f"   Browse: https://huggingface.co/datasets/rhasspy/piper-checkpoints/tree/main/{ckpt_subpath}")
        else:
            local_name = os.path.basename(ckpt_filename)
            local_path = os.path.join(PRETRAINED_DIR, local_name)

            if os.path.exists(local_path):
                print(f"Pretrained checkpoint already downloaded: {local_path}")
                pretrained_ckpt_path = local_path
            else:
                print(f"Downloading: {ckpt_filename}")
                downloaded = hf_hub_download(
                    repo_id=repo_id,
                    filename=ckpt_filename,
                    repo_type="dataset",
                    local_dir=PRETRAINED_DIR,
                    local_dir_use_symlinks=False,
                )
                # hf_hub_download may place file in a subdirectory structure;
                # move it to PRETRAINED_DIR root for simpler path handling
                if os.path.exists(downloaded) and downloaded != local_path:
                    import shutil
                    shutil.move(downloaded, local_path)
                pretrained_ckpt_path = local_path
                print(f"Downloaded to: {pretrained_ckpt_path}")

    except Exception as e:
        print(f"ERROR downloading checkpoint: {e}")
        print("   Training will proceed from scratch if no checkpoint is available.")

# Save to project state for Cell 22
state.setdefault("piper", {})
state["piper"]["pretrained_checkpoint"] = pretrained_ckpt_path
write_json(PROJECT_JSON_PATH, state)

if pretrained_ckpt_path:
    size_mb = os.path.getsize(pretrained_ckpt_path) / (1024 * 1024)
    print(f"\nPretrained checkpoint ready ({size_mb:.0f} MB)")
else:
    print("\nNo pretrained checkpoint available. Training will start from scratch.")

In [ ]:
# ============================================================
# @title 📐 Preprocess Dataset (required before training)
# ============================================================
# @markdown Piper training requires preprocessing: converting wav+metadata
# @markdown into phoneme IDs, audio tensors (.pt files), config.json, and dataset.jsonl.
# @markdown This step runs `piper_train.preprocess` on your generated dataset.

import os

train_src = os.path.join(PIPER_REPO_DIR, "src", "python")

# Map our LANGUAGE setting to an espeak-ng voice code for phonemization
LANG_TO_ESPEAK = {
    "English": "en-us",
    "Chinese": "zh",
    "Japanese": "ja",
    "Korean": "ko",
    "German": "de",
    "French": "fr",
    "Russian": "ru",
    "Portuguese": "pt",
    "Spanish": "es",
    "Italian": "it",
    "Auto": "en-us",  # fallback
}
espeak_voice = LANG_TO_ESPEAK.get(LANGUAGE, "en-us")

# Check if preprocessing has already been done
config_json = os.path.join(DATASET_DIR, "config.json")
dataset_jsonl = os.path.join(DATASET_DIR, "dataset.jsonl")

if os.path.exists(config_json) and os.path.exists(dataset_jsonl):
    print("✅ Preprocessing already done. Found:")
    print(f"   config.json: {config_json}")
    print(f"   dataset.jsonl: {dataset_jsonl}")
    print("   (Delete these files to force re-preprocessing)")
else:
    print(f"📐 Preprocessing dataset for language '{espeak_voice}'...")
    print(f"   Input dir: {DATASET_DIR}")
    print(f"   Sample rate: {OUTPUT_SAMPLE_RATE}")

    preprocess_cmd = (
        f"cd '{train_src}' && "
        f"python3 -m piper_train.preprocess "
        f"--language '{espeak_voice}' "
        f"--input-dir '{DATASET_DIR}' "
        f"--output-dir '{DATASET_DIR}' "
        f"--dataset-format ljspeech "
        f"--single-speaker "
        f"--sample-rate {OUTPUT_SAMPLE_RATE}"
    )

    print(f"\nRunning: {preprocess_cmd}\n")
    !bash -lc "$preprocess_cmd"

    # Verify output
    if os.path.exists(config_json):
        print(f"\n✅ Preprocessing complete.")
        print(f"   config.json: {config_json}")
    else:
        print(f"\n❌ Preprocessing may have failed — config.json not found.")
        print(f"   Check the output above for errors.")

    if os.path.exists(dataset_jsonl):
        # Count entries
        with open(dataset_jsonl, "r") as f:
            n_entries = sum(1 for _ in f)
        print(f"   dataset.jsonl: {n_entries} entries")
    else:
        print(f"   ❌ dataset.jsonl not found.")


In [ ]:
# ============================================================
# @title 🧹 Free GPU Memory (unload Qwen before Piper training)
# ============================================================
import gc

# Delete the Qwen model and voice clone prompt — we're done generating
if 'model' in dir() and model is not None:
    del model
    print("🗑️ Qwen model deleted.")

if 'voice_clone_prompt' in dir() and voice_clone_prompt is not None:
    del voice_clone_prompt
    print("🗑️ Voice clone prompt deleted.")

## Monitor Training with TensorBoard

The Piper training process uses PyTorch Lightning, which automatically generates logs compatible with TensorBoard. These logs are stored in a `lightning_logs` subdirectory within your `default_root_dir` (which is stored in the `train_out` variable from the previous cell).

You can launch TensorBoard in a new output pane below to visualize metrics like loss, learning rate, and other training progress indicators in real-time.

Keep in mind that TensorBoard will only show data that has already been written to the logs. If training is still ongoing, new data will appear as it's logged.

In [ ]:
# @title 1. Launch TensorBoard

# Load and launch TensorBoard
%load_ext tensorboard
%tensorboard --logdir "{train_out}"

In [ ]:
# ============================================================
# @title Train / Resume Piper
# ============================================================

import re
import time
import gc

# NOTE:
# - Piper training fine-tunes a VITS TTS voice model.
# - Qwen3-TTS above is only a *data generator* (no Qwen training here).

# Define paths (re-defining here to ensure self-contained execution)
train_src = os.path.join(PIPER_REPO_DIR, "src", "python")
train_out = ensure_dir(os.path.join(
    PIPER_DIR, "training_runs", time.strftime("%Y%m%d_%H%M%S")
))

# Clear memory and GPU in case we resume training
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.synchronize()
    free_mem = torch.cuda.mem_get_info()[0] / 1024**3
    total_mem = torch.cuda.mem_get_info()[1] / 1024**3
    print(f"\n✅ GPU memory freed: {free_mem:.1f} / {total_mem:.1f} GB available")

# ---- PATCH: Add Enhanced Monitoring (GPU, LR) ----
# We patch piper_train/__main__.py to inject DeviceStatsMonitor and LearningRateMonitor
piper_main = os.path.join(train_src, "piper_train", "__main__.py")
if os.path.exists(piper_main):
    with open(piper_main, "r") as f:
        code = f.read()

    if "DeviceStatsMonitor" not in code:
        print("🔧 Patching piper_train for enhanced monitoring (GPU, LR)...")

        # 1. Add Imports
        if "from pytorch_lightning.callbacks import DeviceStatsMonitor" not in code:
            code = code.replace("import pytorch_lightning as pl",
                                "import pytorch_lightning as pl\nfrom pytorch_lightning.callbacks import DeviceStatsMonitor, LearningRateMonitor")

        # 2. Inject Callbacks
        # Look for: trainer = pl.Trainer.from_argparse_args(args, ...)
        pattern = r"(trainer\s*=\s*pl\.Trainer\.from_argparse_args\s*\()"
        match = re.search(pattern, code)

        if match:
            # Determine indentation from the line found
            line_start_idx = code.rfind('\n', 0, match.start()) + 1
            indent = code[line_start_idx:match.start()]
            if not indent.strip(): indent = "    "

            # Create the patch block
            patch_block = (
                f"\n{indent}# PATCH: Add monitoring callbacks\n"
                f"{indent}if 'callbacks' not in locals(): callbacks = []\n"
                f"{indent}callbacks.append(DeviceStatsMonitor())\n"
                f"{indent}callbacks.append(LearningRateMonitor(logging_interval='step'))\n"
                f"{indent}" # maintain spacing for original line
            )

            # Insert the patch block BEFORE the trainer init
            code = code[:match.start()] + patch_block + code[match.start():]

            # 3. Ensure callbacks arg is passed
            # Re-find the call (position shifted)
            match = re.search(pattern, code)
            # Check next ~300 chars for 'callbacks='
            lookahead = code[match.end():match.end()+300]
            if "callbacks=" not in lookahead:
                # Insert 'callbacks=callbacks, ' right after the opening parenthesis
                code = code[:match.end()] + "callbacks=callbacks, " + code[match.end():]

            with open(piper_main, "w") as f:
                f.write(code)
            print("   ✅ Monitoring callbacks added.")
        else:
            print("   ⚠️ Could not find Trainer init line to patch. Monitoring might be limited.")
    else:
        print("   ✅ piper_train already patched for monitoring.")

# ---- Verify preprocessing was done ----
config_json = os.path.join(DATASET_DIR, "config.json")
dataset_jsonl = os.path.join(DATASET_DIR, "dataset.jsonl")
if not os.path.exists(config_json) or not os.path.exists(dataset_jsonl):
    raise FileNotFoundError(
        "Preprocessing not done! Run the 'Preprocess Dataset' cell first.\n"
        f"  Missing: {'config.json' if not os.path.exists(config_json) else ''} "
        f"{'dataset.jsonl' if not os.path.exists(dataset_jsonl) else ''}"
    )

# ---- 3-tier checkpoint priority ----
resume_flag = ""
last_ckpt = state.get("piper", {}).get("last_checkpoint")
pretrained_ckpt = state.get("piper", {}).get("pretrained_checkpoint")

if PIPER_RESUME and last_ckpt and os.path.exists(last_ckpt):
    # Priority 1: Resume interrupted training
    resume_flag = f"--resume_from_checkpoint '{last_ckpt}'"
    print("Resuming interrupted training from:", last_ckpt)
elif pretrained_ckpt and os.path.exists(pretrained_ckpt):
    # Priority 2: Fine-tune from pretrained checkpoint
    stripped_ckpt = pretrained_ckpt.replace('.ckpt', '.weights.ckpt')
    print("Preparing checkpoint for fine-tuning (stripping optimizer states)...")
    _ckpt = torch.load(pretrained_ckpt, map_location='cpu', weights_only=False)
    _ckpt['optimizer_states'] = []
    _ckpt['lr_schedulers'] = []
    _ckpt['epoch'] = 0
    _ckpt['global_step'] = 0
    torch.save(_ckpt, stripped_ckpt)
    _orig_mb = os.path.getsize(pretrained_ckpt) / 1024**2
    _strip_mb = os.path.getsize(stripped_ckpt) / 1024**2
    print(f"   {_orig_mb:.0f} MB -> {_strip_mb:.0f} MB (optimizer states removed)")
    del _ckpt
    import gc; gc.collect()
    resume_flag = f"--resume_from_checkpoint '{stripped_ckpt}'"
    print("Fine-tuning from pretrained checkpoint:", pretrained_ckpt)
else:
    # Priority 3: Training from scratch
    print("WARNING: No checkpoint available. Training from scratch.")

# Determine accelerator
accelerator = "gpu" if (DEVICE == "cuda" and torch.cuda.is_available()) else "cpu"

# Reduce PyTorch memory fragmentation
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
os.environ['JAX_PLATFORMS'] = 'cpu'

# Use Tensor Cores on L4/A100
if torch.cuda.is_available():
    torch.set_float32_matmul_precision('medium')

# Pre-flight VRAM check
if torch.cuda.is_available():
    _free = torch.cuda.mem_get_info()[0] / 1024**3
    _total = torch.cuda.mem_get_info()[1] / 1024**3
    print(f"GPU VRAM: {_free:.1f} / {_total:.1f} GB free")
    if _free < _total - 2.0:
        print(f"   WARNING: {_total - _free:.1f} GB already in use by other processes.")
        print(f"   If training OOMs, reduce PIPER_BATCH_SIZE (currently {PIPER_BATCH_SIZE}).")

# ---- Launch TensorBoard Background Process ----
log_dir = os.path.join(train_out, "lightning_logs")
print(f"\n📊 Launching TensorBoard in background... monitor at {log_dir}")
get_ipython().system_raw(f"tensorboard --logdir '{train_out}' --port 6006 &")

# Build command
cmd_parts = [
    f"cd '{train_src}'",
    "&&",
    "JAX_PLATFORMS=cpu",
    "XLA_PYTHON_CLIENT_PREALLOCATE=false",
    "TF_FORCE_GPU_ALLOW_GROWTH=true",
    "PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True",
    "python -m piper_train",
    f"--dataset-dir '{DATASET_DIR}'",
    f"--accelerator '{accelerator}'",
    "--devices 1",
    f"--batch-size {PIPER_BATCH_SIZE}",
    f"--validation-split {PIPER_VALIDATION_SPLIT}",
    "--num-test-examples 5",
    f"--max_epochs {PIPER_MAX_EPOCHS}",
    f"--checkpoint-epochs {PIPER_CHECKPOINT_EVERY}",
    f"--precision {PIPER_PRECISION}",
    "--max-phoneme-ids 300",
    f"--default_root_dir '{train_out}'",
]

if resume_flag:
    cmd_parts.append(resume_flag)

cmd = " ".join(cmd_parts)

print("Running training command:\n", cmd)
!bash -lc "$cmd"

# ---- Find newest checkpoint ----
ckpt_dir = os.path.join(train_out, "checkpoints")
newest_ckpt = None
if os.path.exists(ckpt_dir):
    ckpts = sorted(
        [os.path.join(ckpt_dir, f) for f in os.listdir(ckpt_dir) if f.endswith(".ckpt")],
        key=os.path.getmtime, reverse=True,
    )
    if ckpts:
        newest_ckpt = ckpts[0]

# Also check lightning_logs
lightning_ckpt_dir = os.path.join(train_out, "lightning_logs")
if newest_ckpt is None and os.path.exists(lightning_ckpt_dir):
    for root, dirs, fnames in os.walk(lightning_ckpt_dir):
        for fn in fnames:
            if fn.endswith(".ckpt"):
                candidate = os.path.join(root, fn)
                if newest_ckpt is None or os.path.getmtime(candidate) > os.path.getmtime(newest_ckpt):
                    newest_ckpt = candidate

state.setdefault("piper", {})
state["piper"].update({
    "last_train_out": train_out,
    "last_checkpoint": newest_ckpt,
    "last_run_unix": int(time.time()),
})
state["history"].append({
    "t": int(time.time()),
    "event": "piper_train",
    "train_out": train_out,
    "checkpoint": newest_ckpt,
})
write_json(PROJECT_JSON_PATH, state)

if newest_ckpt:
    print("Training complete!")
    print("   Output dir:", train_out)
    print("   Newest checkpoint:", newest_ckpt)
else:
    print("WARNING: Training finished but no checkpoint was found.")
    print("   Output dir:", train_out)
    print("   Check the training output above for errors.")

In [ ]:
# @title 📊 Programmatic Log Analysis
# @markdown Run this cell to print a text summary of the latest training metrics from the logs.
# @markdown This is useful if the TensorBoard UI is slow or not loading.

import os
import glob
import pandas as pd
from IPython.display import display
from tensorboard.backend.event_processing.event_accumulator import EventAccumulator

def get_latest_log_dir(base_dir):
    log_root = os.path.join(base_dir, "lightning_logs")
    if not os.path.exists(log_root):
        return None
    # Get all version directories
    versions = sorted([
        d for d in glob.glob(os.path.join(log_root, "version_*"))
        if os.path.isdir(d)
    ], key=os.path.getmtime)

    return versions[-1] if versions else None

def analyze_logs(train_base_dir):
    log_dir = get_latest_log_dir(train_base_dir)
    if not log_dir:
        print(f"❌ No logs found in {train_base_dir}")
        return

    print(f"📂 Reading logs from: {os.path.basename(log_dir)}")

    # Find tfevents file
    event_files = glob.glob(os.path.join(log_dir, "events.out.tfevents.*"))
    if not event_files:
        print("❌ No event file found.")
        return

    latest_event = max(event_files, key=os.path.getmtime)

    try:
        ea = EventAccumulator(latest_event)
        ea.Reload()

        tags = ea.Tags()['scalars']
        metrics = {}

        # Extended metrics to look for
        keys = [
            'loss_gen_all', 'loss_mel', 'loss_kl', 'epoch',
            'lr-Adam', 'lr',  # Learning rate variations
            'hp_metric',      # Hyperparams
        ]

        # Add GPU stats patterns (DeviceStatsMonitor)
        gpu_patterns = ['gpu_id:0/memory.used', 'gpu_id:0/utilization.gpu']

        # First pass: Exact matches
        for k in keys:
            if k in tags:
                events = ea.Scalars(k)
                if events:
                    metrics[k] = events[-1].value

        # Second pass: Partial matches for GPU/LR if exact names vary
        for tag in tags:
            if 'lr' in tag.lower() and 'Adam' in tag and tag not in metrics:
                 metrics['Learning Rate'] = ea.Scalars(tag)[-1].value
            for pat in gpu_patterns:
                if pat in tag:
                    clean_name = pat.split('/')[-1].replace('.', ' ').title()
                    metrics[clean_name] = ea.Scalars(tag)[-1].value

        if metrics:
            print("\n📈 Latest Status:")

            # Group 1: Main Training Metrics
            print("  [Training]")
            if 'epoch' in metrics: print(f"   Epoch: {metrics['epoch']:.0f}")
            if 'loss_gen_all' in metrics: print(f"   Loss Gen Total: {metrics['loss_gen_all']:.4f}")
            if 'loss_kl' in metrics: print(f"   Loss KL: {metrics['loss_kl']:.4f}")
            if 'loss_mel' in metrics: print(f"   Loss Mel: {metrics['loss_mel']:.4f}")

            # Group 2: Performance / Stats
            print("\n  [Performance]")
            if 'Learning Rate' in metrics: print(f"   Learning Rate: {metrics['Learning Rate']:.2e}")
            if 'Gpu Id:0/Memory Used' in metrics: print(f"   GPU Mem: {metrics['Gpu Id:0/Memory Used']:.0f} MB")
            if 'Gpu Id:0/Utilization Gpu' in metrics: print(f"   GPU Util: {metrics['Gpu Id:0/Utilization Gpu']:.0f}%")

            if 'loss_mel' in metrics:
                mel = metrics['loss_mel']
                print("\n💡 Tip: 'loss_mel' is the best proxy for voice quality.")
                if mel > 20: print("   > 20: Audio likely noisy/robotic")
                elif mel > 15: print("   15-20: Becoming intelligible")
                elif mel < 15: print("   < 15: Good quality emerging")

        else:
            print("⚠️ Logs found, but no scalar metrics written yet.")

    except Exception as e:
        print(f"⚠️ Error reading logs: {e}")

# Determine which directory to check
target_dir = None
if 'train_out' in locals() and os.path.exists(train_out):
    target_dir = train_out
else:
    # Try finding it from state
    try:
        _st = read_json(PROJECT_JSON_PATH)
        target_dir = _st.get("piper", {}).get("last_train_out")
    except:
        pass

if target_dir and os.path.exists(target_dir):
    analyze_logs(target_dir)
else:
    print("Could not find training output directory. Run training first.")

In [ ]:
# ============================================================
# @title 📌 Project Status Summary
# ============================================================

state = read_json(PROJECT_JSON_PATH)

print("=" * 50)
print(f"Project: {state.get('project_name')} ({state.get('project_slug')})")
print(f"Language: {state.get('language')}")
print(f"Qwen model: {state.get('qwen_model_id')}")
print(f"Reference audio: {state.get('ref_audio', {}).get('path')}")
print(f"Ref text: {state.get('ref_audio', {}).get('ref_text', '(none)')}")
print(f"Ref text source: {state.get('ref_audio', {}).get('ref_text_source', 'unknown')}")
print(f"Dataset dir: {DATASET_DIR}")
print(f"  wavs/: {WAVS_DIR}")
print(f"  metadata: {METADATA_PATH}")
print(f"  Total generated: {state.get('dataset', {}).get('total_generated', '?')}")
print(f"Piper:")
print(f"  Last run: {state.get('piper', {}).get('last_train_out', 'none')}")
print(f"  Last ckpt: {state.get('piper', {}).get('last_checkpoint', 'none')}")
print("=" * 50)

In [ ]:
# ============================================================
# @title 🔈 Test Your Piper Voice
# ============================================================
# @markdown ---
# @markdown ### Type a sentence to hear your cloned voice!
# @markdown Export the trained checkpoint to ONNX, then synthesize and play.

TEST_TEXT = "Hello! This is a test of my cloned voice using Piper text to speech."  # @param {type:"string"}
TEST_TEXT_2 = "The quick brown fox jumps over the lazy dog."  # @param {type:"string"}
TEST_TEXT_3 = ""  # @param {type:"string"}

# @markdown ---
# @markdown ### Export settings
FORCE_RE_EXPORT = False  # @param {type:"boolean"}

import os
import glob
import subprocess
import torch
import soundfile as sf
from IPython.display import Audio, display

# ---- PATCH: Fix ONNX Export Stability ----
# 1. Patch piper_train/vits/transforms.py (Discriminant & Shape Checks)
transforms_path = os.path.join(PIPER_REPO_DIR, "src", "python", "piper_train", "vits", "transforms.py")
if os.path.exists(transforms_path):
    with open(transforms_path, "r") as f:
        code = f.read()

    # Fix 1: Discriminant assertion
    target_line = "assert (discriminant >= 0).all(), discriminant"
    if target_line in code:
        print("🔧 Patching transforms.py: discriminant safety...")
        new_line = "discriminant = torch.clamp(discriminant, min=0) # patched safety"
        code = code.replace(target_line, new_line)
        with open(transforms_path, "w") as f:
            f.write(code)
        print("   ✅ Applied.")

# 2. Patch piper_train/export_onnx.py (Dummy Input Length & Legacy Exporter)
export_script_path = os.path.join(PIPER_REPO_DIR, "src", "python", "piper_train", "export_onnx.py")
if os.path.exists(export_script_path):
    with open(export_script_path, "r") as f:
        export_code = f.read()

    # Patch A: Increase dummy_input_length to 256 (very safe) to prevent dimension collapse
    if "dummy_input_length = 50" in export_code:
        print("🔧 Patching export_onnx.py: increasing dummy input size (50 -> 256)...")
        export_code = export_code.replace("dummy_input_length = 50", "dummy_input_length = 256")
        with open(export_script_path, "w") as f:
            f.write(export_code)
        print("   ✅ Applied size increase.")
    elif "dummy_input_length = 100" in export_code:
        print("🔧 Patching export_onnx.py: increasing dummy input size (100 -> 256)...")
        export_code = export_code.replace("dummy_input_length = 100", "dummy_input_length = 256")
        with open(export_script_path, "w") as f:
            f.write(export_code)
        print("   ✅ Applied size increase.")

    # Patch B: Force legacy exporter (dynamo=False) to avoid torch._check errors
    # This is critical for VITS on PyTorch 2.x
    if "dynamo=False" not in export_code:
        print("🔧 Patching export_onnx.py: forcing legacy exporter (dynamo=False)...")
        # We inject it into the opset_version line which is standard in Piper code
        if "opset_version=OPSET_VERSION" in export_code:
            export_code = export_code.replace(
                "opset_version=OPSET_VERSION",
                "opset_version=OPSET_VERSION, dynamo=False"
            )
            with open(export_script_path, "w") as f:
                f.write(export_code)
            print("   ✅ Applied legacy export enforcement.")
        else:
            print("   ⚠️ Could not find injection point for dynamo=False.")
    else:
        print("   ℹ️ Legacy exporter already enforced.")

# ---- Locate the latest checkpoint ----
state = read_json(PROJECT_JSON_PATH)
ckpt_path = state.get("piper", {}).get("last_checkpoint")

if not ckpt_path or not os.path.exists(ckpt_path):
    # Try to find any checkpoint in the training runs
    search_pattern = os.path.join(PIPER_DIR, "training_runs", "**", "*.ckpt")
    all_ckpts = glob.glob(search_pattern, recursive=True)
    if all_ckpts:
        ckpt_path = max(all_ckpts, key=os.path.getmtime)
        print(f"Found checkpoint: {ckpt_path}")
    else:
        raise FileNotFoundError(
            "No training checkpoint found. Run the training cell first!"
        )
else:
    print(f"Using checkpoint: {ckpt_path}")

# ---- Paths for export ----
MODEL_DIR = ensure_dir(os.path.join(OUTPUTS_DIR, "piper_model"))
ONNX_PATH = os.path.join(MODEL_DIR, f"{project_slug}.onnx")
ONNX_JSON_PATH = ONNX_PATH + ".json"
TEST_OUTPUT_DIR = ensure_dir(os.path.join(OUTPUTS_DIR, "test_audio"))

# ---- Find config.json from training ----
# Piper generates config.json in the dataset dir during preprocessing,
# or in the training output dir
config_json = None
for candidate in [
    os.path.join(DATASET_DIR, "config.json"),
    os.path.join(state.get("piper", {}).get("last_train_out", ""), "config.json"),
]:
    if os.path.exists(candidate):
        config_json = candidate
        break

# ---- Step 1: Export to ONNX ----
train_py = os.path.join(PIPER_REPO_DIR, "src", "python")

if not os.path.exists(ONNX_PATH) or FORCE_RE_EXPORT:
    print("📦 Exporting checkpoint to ONNX...")
    print(f"   Checkpoint: {ckpt_path}")
    print(f"   Output: {ONNX_PATH}")

    export_cmd = (
        f"cd '{train_py}' && "
        f"python3 -m piper_train.export_onnx "
        f"'{ckpt_path}' "
        f"'{ONNX_PATH}'"
    )
    result = subprocess.run(
        ["bash", "-lc", export_cmd],
        capture_output=True, text=True
    )
    if result.returncode != 0:
        print("❌ Export failed!")
        print("STDOUT:", result.stdout[-500:] if result.stdout else "")
        print("STDERR:", result.stderr[-500:] if result.stderr else "")
        raise RuntimeError("ONNX export failed. Check the error above.")

    # Copy config.json alongside the ONNX model
    if config_json and os.path.exists(config_json):
        import shutil
        shutil.copy2(config_json, ONNX_JSON_PATH)
        print(f"   Config: {ONNX_JSON_PATH}")

    print("✅ ONNX export complete!")
else:
    print(f"✅ ONNX model already exists: {ONNX_PATH}")
    if not os.path.exists(ONNX_JSON_PATH) and config_json:
        import shutil
        shutil.copy2(config_json, ONNX_JSON_PATH)

# Verify files exist
if not os.path.exists(ONNX_PATH):
    raise FileNotFoundError(f"ONNX model not found: {ONNX_PATH}")
if not os.path.exists(ONNX_JSON_PATH):
    print("⚠️ Warning: config JSON not found. Piper may not work without it.")
    print(f"   Expected at: {ONNX_JSON_PATH}")
    print(f"   Try copying config.json from your dataset/training dir manually.")

# ---- Step 2: Install piper-tts CLI if needed ----
try:
    subprocess.run(["piper", "--version"], capture_output=True, check=True)
    print("✅ piper-tts CLI available.")
except (subprocess.CalledProcessError, FileNotFoundError):
    print("📦 Installing piper-tts CLI...")
    subprocess.run(
        ["pip", "-q", "install", "piper-tts"],
        capture_output=True, check=True
    )
    print("✅ piper-tts CLI installed.")

# ---- Step 3: Synthesize test sentences ----
test_sentences = [s.strip() for s in [TEST_TEXT, TEST_TEXT_2, TEST_TEXT_3] if s.strip()]

if not test_sentences:
    print("⚠️ No test sentences provided. Enter text above and re-run.")
else:
    print(f"\n🎤 Synthesizing {len(test_sentences)} test sentence(s)...\n")

    for i, sentence in enumerate(test_sentences, 1):
        wav_out = os.path.join(TEST_OUTPUT_DIR, f"test_{i}.wav")

        synth_cmd = (
            f"echo '{sentence}' | "
            f"piper -m '{ONNX_PATH}' --output_file '{wav_out}'"
        )
        result = subprocess.run(
            ["bash", "-c", synth_cmd],
            capture_output=True, text=True
        )

        if result.returncode != 0 or not os.path.exists(wav_out):
            print(f"  ❌ Sentence {i} failed:")
            print(f"     {result.stderr[:300] if result.stderr else 'Unknown error'}")
            continue

        # Get duration
        try:
            info = sf.info(wav_out)
            duration = info.duration
        except Exception:
            duration = 0

        print(f"  🔊 [{i}] ({duration:.1f}s): \"{sentence[:80]}{'...' if len(sentence) > 80 else ''}\"")

        # Play inline in Colab
        display(Audio(wav_out, autoplay=(i == 1)))

    print(f"\n✅ Test audio saved to: {TEST_OUTPUT_DIR}")

# ---- Update state ----
state["piper"]["onnx_path"] = ONNX_PATH
state["piper"]["onnx_json_path"] = ONNX_JSON_PATH
state["history"].append({
    "t": int(time.time()),
    "event": "test_voice",
    "onnx_path": ONNX_PATH,
    "num_sentences": len(test_sentences),
})
write_json(PROJECT_JSON_PATH, state)

print(f"\n📁 Model files (copy these for use with Piper anywhere):")
print(f"   ONNX model:  {ONNX_PATH}")
print(f"   Config JSON:  {ONNX_JSON_PATH}")